In [18]:
import dspy
from typing import Literal, List
dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)


class CLassifyDomain(dspy.Signature):
        """
        Classify the study design described in the abstract.

        Available study-design labels:
            "randomized_controlled_trial", "nonrandomized_controlled_trial",
            "prospective_cohort", "retrospective_cohort", "case_control",
            "cross_sectional", "case_series", "case_report",
            "diagnostic_accuracy_study", "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development", "in_vitro", "animal_model",
            "imaging_only", "conference_abstract_or_poster",
            "other_or_unclear"

        **Primary design selection rules:**
        - Randomized allocation → randomized_controlled_trial
        - Nonrandomized comparator groups → nonrandomized_controlled_trial
        - Prospective follow-up of a group → prospective_cohort
        - Retrospective chart/registry review → retrospective_cohort
        - Explicit “cases vs controls” comparison → case_control
        - Single time-point measurement/survey/prevalence → cross_sectional
        - ≥2 patients without a control group → case_series
        - Single patient → case_report
        - Test/validation of diagnostic performance → diagnostic_accuracy_study
        - Systematic review or meta-analysis → systematic_review_or_meta_analysis
        - No primary data (guidelines, commentary, editorial) → guideline_or_editorial_or_commentary
        - Methods/assay development without clinical outcomes → methods_or_assay_development
        - In vitro experiments → in_vitro
        - Animal experiments → animal_model
        - Imaging-only analyses without clinical outcomes → imaging_only
        - Conference abstracts/posters → conference_abstract_or_poster
        - Anything unclear or mixed → other_or_unclear

        **Secondary design rules:**
        - secondary_designs must be a JSON array.
        - Choose 0–3 additional labels if they meaningfully apply.
        - Use only labels from the primary-design list.
        - Use [] if none apply.

        **Task:**
        Read the abstract and output:
            (1) the single best-fitting primary_design
            (2) an optional list (0–3 items) of secondary_designs
        """
    
        abstract: str = dspy.InputField(
            desc="The Abstract text to classify into themes"
        )
        primary_design: Literal[
            "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
        ] = dspy.OutputField(desc="The primary design classification")
        secondary_designs: List[
            Literal[
                      "randomized_controlled_trial",
            "nonrandomized_controlled_trial",
            "prospective_cohort",
            "retrospective_cohort",
            "case_control",
            "cross_sectional",
            "case_series",
            "case_report",
            "diagnostic_accuracy_study",
            "systematic_review_or_meta_analysis",
            "guideline_or_editorial_or_commentary",
            "methods_or_assay_development",
            "in_vitro",
            "animal_model",
            "imaging_only",
            "conference_abstract_or_poster",
            "other_or_unclear",
            ]
        ] = dspy.OutputField(desc="list of 0-3 secondary design classifications")


        

In [7]:
def create_DSPy_example(data):
    gold_standard = []
    for row in data:
        gold_standard.append(
            dspy.Example(
                # context = CONTEXT,
                abstract=row['abstract'],
                primary_design=row['primary_design'],
                secondary_designs=row['secondary_designs'],
            ).with_inputs("abstract"),
        )
    return gold_standard

In [8]:
import json
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_train.json", "r") as f:
    train = json.load(f)
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_val.json", "r") as f:
    val = json.load(f)
with open("../../validation/domain_classification/classification_results_ensembled_domain_classification_test.json", "r") as f:
    test = json.load(f)
gold_standard_train = create_DSPy_example(train)
gold_standard_val = create_DSPy_example(val)
gold_standard_test = create_DSPy_example(test)

In [11]:
def gepa_feedback_metric(gold: dspy.Example,
                         pred: dspy.Prediction,
                         trace=None,
                         pred_name=None,
                         pred_trace=None):
    # simple exact-match score on the decision
    pred_primary_design = getattr(pred, "primary_design", None)
    gold_primary_design = getattr(gold, "primary_design", None)
    
    score = 0
    if pred_primary_design == gold_primary_design:
        score = 1.0
    else:
        score = 0.0
    
    # brief feedback that GEPA can reflect on
    if score == 1.0:
        fb = "Decision matches gold. Keep citing PICOS elements clearly."
    else:
        # Assuming your student output has 'reasoning', 'classification', and 'confidence'
# and your gold data has 'classification' and 'abstract' (as context)

        fb = (
            f"Decision does not match gold. "
            f"Predicted: {pred_primary_design}. "
            f"Gold: {gold_primary_design}. "
            f"Review the abstract and ensure correct classification."
        )

    # Return only the score for GEPA
    return dspy.Prediction(score=score, feedback=fb)

In [13]:
import json
import os

student_llm_string = 'openrouter/google/gemini-2.0-flash-001'
screener_results_path = "../../classifier/domain_classification/classification_results_gemini_groundtruth.json"
results_path = "../../results/domain_classification/classification_results_gemini_original.json"
results_path_save = "../../results/domain_classification/classification_results_gemini_original.jsonl"

API_KEY = os.getenv("openrouter_api_key")
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 350000,
)
dspy.configure(lm=student_lm)


In [19]:
gepa = dspy.GEPA(
    metric=gepa_feedback_metric,      # your feedback metric
    reflection_lm=student_lm,  
    # only the reflector is stochastic
    auto = 'medium',
    reflection_minibatch_size=5,
    use_merge=True,
    max_merge_invocations=10,
    track_stats=True,
    # skip_perfect_score=False,
)

compiled_screener = gepa.compile(
    student=dspy.ChainOfThought(CLassifyDomain),
    trainset=gold_standard_train, # minimal viable setup
    valset=gold_standard_val,
)
cost = sum([x['cost'] for x in student_lm.history if x['cost'] is not None])  # cost in USD, as calculated by LiteLLM for certain providers
print(cost)

# --- save and load as before ---
compiled_screener.save(path=screener_results_path)
# same lm used for a minimal setup in this example

2025/11/25 10:33:32 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 825 metric calls of the program. This amounts to 7.86 full evals on the train+val set.
2025/11/25 10:33:32 INFO dspy.teleprompt.gepa.gepa: Using 27 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
2025/11/25 10:33:38 INFO dspy.evaluate.evaluate: Average Metric: 25.0 / 27 (92.6%)
2025/11/25 10:33:38 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.9259259259259259
2025/11/25 10:33:38 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.9259259259259259


Average Metric: 5.00 / 5 (100.0%): 100%|██████████| 5/5 [00:01<00:00,  3.81it/s]

2025/11/25 10:33:39 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 10:33:39 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2025/11/25 10:33:39 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
2025/11/25 10:33:39 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.9259259259259259



Average Metric: 5.00 / 5 (100.0%): 100%|██████████| 5/5 [00:01<00:00,  3.26it/s]

2025/11/25 10:33:41 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 10:33:41 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2025/11/25 10:33:41 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
2025/11/25 10:33:41 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 0.9259259259259259


...

2025/11/25 10:39:30 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 5 (100.0%)
2025/11/25 10:39:30 INFO dspy.teleprompt.gepa.gepa: Iteration 103: All subsample scores perfect. Skipping.
2025/11/25 10:39:30 INFO dspy.teleprompt.gepa.gepa: Iteration 103: Reflective mutation did not propose a new candidate
2025/11/25 10:39:30 INFO dspy.teleprompt.gepa.gepa: Iteration 104: Selected program 7 score: 1.0



Average Metric: 4.00 / 5 (80.0%): 100%|██████████| 5/5 [00:02<00:00,  2.37it/s] 

2025/11/25 10:39:32 INFO dspy.evaluate.evaluate: Average Metric: 4.0 / 5 (80.0%)


2025/11/25 10:39:42 INFO dspy.teleprompt.gepa.gepa: Iteration 104: Proposed new text for predict: You are an expert study-design classifier specializing in Lyme disease research. You will be provided with an abstract of a research paper and must classify the study design based on the content of the abstract. Your classification should leverage your understanding of Lyme disease interventions, diagnostic criteria, and the nuances of studies related to this condition. Pay close attention to phrases hinting at specific study designs.

Available study-design labels:
    "randomized_controlled_trial", "nonrandomized_controlled_trial",
    "prospective_cohort", "retrospective_cohort", "case_control",
    "cross_sectional", "case_series", "case_report",
    "diagnostic_accuracy_study", "systematic_review_or_meta_analysis",
    "guideline_or_editorial_or_commentary",
    "methods_or_assay_development", "in_vitro", "animal_model",
    "imaging_only", "conference_abstract_or_poster",
    "other_

0.8988340000000079


In [20]:
domain_classifier = dspy.ChainOfThought(CLassifyDomain)
domain_classifier.load(path=screener_results_path)
test_results = []
total_accuracy = 0.0
for example in gold_standard_test:
    pred = domain_classifier(abstract=example.abstract)
    if pred.primary_design == example.primary_design:
        accuracy = 1.0
    else:
        accuracy = 0.0
    test_results.append({
        "abstract": example.abstract,
        "primary_design": pred.primary_design,
        "secondary_designs": pred.secondary_designs,
        "accuracy": accuracy,
    })
    total_accuracy += accuracy
print(f"Test Accuracy: {total_accuracy / len(gold_standard_test):.2f}")

Test Accuracy: 0.96
